In [7]:
import pandas as pd
from thefuzz import process, fuzz

In [36]:
# Increase the width of the display and show all columns
pd.set_option('display.max_columns', None)  
pd.set_option('display.width', 1000)        
pd.set_option('display.expand_frame_repr', False) # This prevents the "split" you are seeing

In [49]:
lookup_df = pd.read_csv('../datasets/2_master_data/master_data.csv')
lookup_df

,group_name,parent_company,organization_type
0,Abbey Well,Coca-Cola Company,child_brand
1,Ades,Coca-Cola Company,child_brand
2,Alhambra,Coca-Cola Company,child_brand
3,American,Coca-Cola Company,child_brand
4,Ameyal,Coca-Cola Company,child_brand
...,...,...,...
1899,Wipol,Unilever,child_brand
1900,Xedex,Unilever,child_brand
1901,Zendium,Unilever,child_brand
1902,Zhonghua,Unilever,child_brand


In [50]:
df = pd.read_excel('../datasets/3_input_data/customer_list.xlsx')
print(df.info())
df

<class 'pandas.DataFrame'>
RangeIndex: 170 entries, 0 to 169
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_name      170 non-null    str    
 1   group_name         0 non-null      float64
 2   parent_name        0 non-null      float64
 3   organization_type  0 non-null      float64
dtypes: float64(3), str(1)
memory usage: 5.4 KB
None


,customer_name,group_name,parent_name,organization_type
0,Kraft Mac & Cheese,NaN,NaN,NaN
1,Jell-O,NaN,NaN,NaN
2,Milo,NaN,NaN,NaN
3,Garnier,NaN,NaN,NaN
4,Neutrogena,NaN,NaN,NaN
...,...,...,...,...
165,McDonald,NaN,NaN,NaN
166,KFC,NaN,NaN,NaN
167,MK restaurants,NaN,NaN,NaN
168,Hooray,NaN,NaN,NaN


In [51]:
new_list = df['customer_name'].tolist()
new_list

['Kraft Mac & Cheese',
 'Jell-O',
 'Milo',
 'Garnier',
 'Neutrogena',
 'Pepsi',
 'Milo',
 "Kiehl's",
 'Rexona',
 "Johnson's Baby",
 'Olay',
 "Lay's",
 'Dove',
 "Ben and Jerry's",
 'Kraft Mac & Cheese',
 'Pantene',
 "Ben & Jerry's",
 'Kraft Mac & Cheese',
 'Pantene',
 'Vichy',
 'Rexona',
 'Pantene',
 'Head & Shoulders',
 'Rexona',
 "Lay's",
 'Oreo',
 'Sour Patch Kids',
 'Maybelline',
 'Rexona',
 'Milo',
 'CeraVe',
 "Lay's",
 'Dove',
 'Dove',
 'Kraft Mac & Cheese',
 'Milo',
 'Maybelline',
 "Johnson's Baby",
 'Sprite',
 "Lay's",
 'Pepsi',
 'La Roche Posay',
 'Coca Cola',
 'Oscar Mayer',
 'Dove',
 'Kraft Mac & Cheese',
 'Fanta',
 "Johnson's Baby",
 'Lay',
 "Lay's",
 'Sprite',
 'Oreo',
 'Colgate',
 'Olay',
 'Oral-B',
 'NescafÃ©',
 'Häagen-Dazs',
 'Minute Maid',
 'Maxwell House',
 'Gillette',
 "Johnson's Baby",
 'NescafÃ©',
 'Rexona',
 'Pantene',
 "L'OrÃ©al Paris",
 'KitKat',
 'Doritos',
 "Lay's",
 'Oreo',
 "Johnson's Baby",
 'Murphy Oil Soap',
 "Pond's",
 "L'OrÃ©al Paris",
 'CocaCola',
 'Mi

In [62]:
results = []

for name in new_list:
    print(f"Processing: {name}")
    
    # 1. Calculate scores for all rows in master data
    df_match = lookup_df.copy()
    df_match['ratio'] = df_match['group_name'].apply(
        lambda x: fuzz.token_set_ratio(str(x).lower().strip(), str(name).lower().strip())
    )

    # 2. Sort by ratio (highest first)
    df_match = df_match.sort_values(by=['ratio'], ascending=False)
    
    # 3. Get the top result
    top_row = df_match.iloc[0]
    
    # 4. Check threshold BEFORE assigning (Instruction Requirement)
    if top_row['ratio'] >= 90:
        group_name = top_row['group_name']
        parent_name = top_row['parent_company']
        org_type = top_row['organization_type']
    else:
        # Unknown Classification as per instructions
        group_name = 'Unknown'
        parent_name = 'Unknown'
        org_type = 'Other'

    # Append to our final results list
    results.append({
        "Customer Name": name,
        "Group Name": group_name,
        "Parent Name": parent_name,
        "Organization Type": org_type
    })

# Create the final DataFrame
classified_df = pd.DataFrame(results)

# Display the first few results to verify
print("\nFinal Classification Sample:")
print(classified_df.head())

Processing: Kraft Mac & Cheese
Processing: Jell-O
Processing: Milo
Processing: Garnier
Processing: Neutrogena
Processing: Pepsi
Processing: Milo
Processing: Kiehl's
Processing: Rexona
Processing: Johnson's Baby
Processing: Olay
Processing: Lay's
Processing: Dove
Processing: Ben and Jerry's
Processing: Kraft Mac & Cheese
Processing: Pantene
Processing: Ben & Jerry's
Processing: Kraft Mac & Cheese
Processing: Pantene
Processing: Vichy
Processing: Rexona
Processing: Pantene
Processing: Head & Shoulders
Processing: Rexona
Processing: Lay's
Processing: Oreo
Processing: Sour Patch Kids
Processing: Maybelline
Processing: Rexona
Processing: Milo
Processing: CeraVe
Processing: Lay's
Processing: Dove
Processing: Dove
Processing: Kraft Mac & Cheese
Processing: Milo
Processing: Maybelline
Processing: Johnson's Baby
Processing: Sprite
Processing: Lay's
Processing: Pepsi
Processing: La Roche Posay
Processing: Coca Cola
Processing: Oscar Mayer
Processing: Dove
Processing: Kraft Mac & Cheese
Processin

In [63]:
classified_df

,Customer Name,Group Name,Parent Name,Organization Type
0,Kraft Mac & Cheese,Unknown,Unknown,Other
1,Jell-O,Jell,Kraft Heinz,child_brand
2,Milo,Milo,Nestlé,child_brand
3,Garnier,"Nestlé owns 23.29% of L'Oréal, the world's lar...",Nestlé,parent_brand
4,Neutrogena,Neutrogena,Johnson & Johnson,child_brand
...,...,...,...,...
165,McDonald,Unknown,Unknown,Other
166,KFC,Unknown,Unknown,Other
167,MK restaurants,Unknown,Unknown,Other
168,Hooray,Unknown,Unknown,Other


In [ ]:
d